# 第二次作业 05 定向补采计划与覆盖检查

生成定向补查计划，执行补采并核对原时刻缺失及后续日期覆盖。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

**历史采集流程**：2026年11月的原采集窗口已结束。请结合已保存结果阅读，勿将“全部运行”视为重新采集入口；代码中的日期、截止条件和来源区分原样保留。

[上一份 04](hw02_04_source_audit_and_routes.ipynb) · [下一份 06](hw02_06_time_matrix.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage1_collect",
    "stage1c_parser_v2",
    "stage1e_followups",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 问题1—核心代码14：冻结补采计划与输入核验

In [1]:
# 功能：核对已通过审计的数据版本，读取真实补查计划，将原时刻/站码缺失查询纳入一次复查。
import stage1_collect
import stage1c_parser_v2
import stage1e_followups
for module in (stage1_collect, stage1c_parser_v2, stage1e_followups):
    importlib.reload(module)
followup_plan, plan_info = stage1e_followups.plan_followups()
for key in ["refresh_count", "followup_count", "total_queries", "snapshot_deadline_exclusive"]:
    print(f"{key}: {plan_info[key]}")
display(pd.Series(plan_info["plan_date_counts"], name="query_count").rename_axis("query_date").to_frame())
display(pd.DataFrame(followup_plan)[["query_id", "role", "date", "from_city", "to_city"]].head(6))
print("本段只检查计划，尚未发起新查询。")


refresh_count: 68
followup_count: 436
total_queries: 504
snapshot_deadline_exclusive: 2026-11-13T21:28:30.987887+08:00


,query_count
query_date,
2026-11-18,68
2026-11-19,240
2026-11-20,196


,query_id,role,date,from_city,to_city
0,R_N26_N29,refresh_primary_unknown,2026-11-18,拉萨,西宁
1,R_N26_N01,refresh_primary_unknown,2026-11-18,拉萨,北京
2,R_N26_N03,refresh_primary_unknown,2026-11-18,拉萨,石家庄
3,R_N26_N04,refresh_primary_unknown,2026-11-18,拉萨,太原
4,R_N26_N09,refresh_primary_unknown,2026-11-18,拉萨,上海
5,R_N26_N10,refresh_primary_unknown,2026-11-18,拉萨,南京


本段只检查计划，尚未发起新查询。


## 问题1—核心代码15：按固定版本执行定向补采

前置变量或模块：`stage1e_followups`。

In [3]:
# 功能：优先复查拉萨出发时刻，再完成其余复查和后续日期查询；每请求保存一个检查点。
MAX_QUERIES = 504  # 本次最多请求数；可改100分批，续采时不会重复已完成项
supplement = stage1e_followups.collect(actor="user", max_queries=MAX_QUERIES)
SUPPLEMENT_OUT = Path(supplement["output_dir"])
for key in ["completed_queries", "remaining_queries", "train_records", "completed_refresh_queries",
            "completed_later_date_queries", "unavailable_timetable_records", "unresolved_station_records", "blocking_error"]:
    print(f"{key}: {supplement[key]}")
print("保存位置：", SUPPLEMENT_OUT)
print("matrix_ready:", supplement["matrix_ready"], "（补采后仍需核验时刻与来源一致性）")


补采目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\raw\user_stage1e_20261113_143455_080165
1/504 2026-11-18 拉萨→西宁: partial_timetable_unavailable; valid=0
2/504 2026-11-18 拉萨→北京: partial_timetable_unavailable; valid=0
3/504 2026-11-18 拉萨→石家庄: partial_timetable_unavailable; valid=0
4/504 2026-11-18 拉萨→太原: partial_timetable_unavailable; valid=0
25/504 2026-11-18 太原→西宁: partial_timetable_unavailable; valid=5
50/504 2026-11-18 兰州→成都: partial_timetable_unavailable; valid=17
75/504 2026-11-19 重庆→福州: validated_records; valid=5
100/504 2026-11-19 石家庄→拉萨: partial_timetable_unavailable; valid=0
125/504 2026-11-20 沈阳→南宁: validated_records; valid=1
150/504 2026-11-19 南宁→福州: validated_records; valid=4
175/504 2026-11-19 呼和浩特→海口: no_direct_train_returned_not_unreachable; valid=0
200/504 2026-11-19 广州→香港: validated_records; valid=63
225/504 2026-11-19 武汉→南宁: validated_records; valid=13
250/504 2026-11-19 西宁→长春: validated_records; valid=1
275/504 2026-11-19 郑州→海口: validated_records; valid=2
300/504 20

## 问题1—核心代码16：检查原缺失是否恢复与后续日期覆盖

前置变量或模块：`SUPPLEMENT_OUT`、`supplement`。

In [4]:
# 功能：分别展示来源复查和后续日期查询状态，重点查看拉萨→西宁是否有有效首程时刻。
quality_e = pd.read_csv(SUPPLEMENT_OUT / "supplement_query_quality.csv", encoding="utf-8-sig")
refresh_e = pd.read_csv(SUPPLEMENT_OUT / "refresh_comparison.csv", encoding="utf-8-sig")
later_e = pd.read_csv(SUPPLEMENT_OUT / "later_date_records.csv", encoding="utf-8-sig")
status_e = quality_e.groupby(["role", "status"], dropna=False).size().rename("count").to_frame()
display(status_e)
display(refresh_e.loc[refresh_e["from_city"] == "拉萨",
                      [
                          "primary_query_id", "from_city", "to_city", "before_valid",
                          "after_valid", "before_unknown_time", "after_unknown_time",
                      ]])
date_counts_e = later_e["query_date"].value_counts().sort_index().rename_axis("query_date").to_frame("record_count")
display(date_counts_e)
status_e.to_csv(SUPPLEMENT_OUT / "supplement_status_counts.csv", encoding="utf-8-sig")
date_counts_e.to_csv(SUPPLEMENT_OUT / "later_date_counts.csv", encoding="utf-8-sig")
print("剩余补采请求：", supplement["remaining_queries"])
print("原数据未覆盖；刷新记录和后续日期记录分表保留，等待来源一致性与行程核验。")


count
role                    status                                         
followup_date           no_direct_train_returned_not_unreachable     70
                        not_collected                               119
                        partial_station_metadata_unresolved           1
                        partial_timetable_unavailable                24
                        validated_records                           222
refresh_primary_unknown partial_station_metadata_unresolved           5
                        partial_timetable_unavailable                62
                        validated_records                             1

,primary_query_id,from_city,to_city,before_valid,after_valid,before_unknown_time,after_unknown_time
0,N26_N29,拉萨,西宁,0,0,6,6
1,N26_N01,拉萨,北京,0,0,1,1
2,N26_N03,拉萨,石家庄,0,0,1,1
3,N26_N04,拉萨,太原,0,0,1,1
4,N26_N09,拉萨,上海,0,0,1,1
5,N26_N10,拉萨,南京,0,0,1,1
6,N26_N16,拉萨,郑州,0,0,2,2
7,N26_N17,拉萨,武汉,0,0,1,1
8,N26_N18,拉萨,长沙,0,0,1,1
9,N26_N19,拉萨,广州,0,0,1,1


,record_count
query_date,
2026-11-19,3249
2026-11-20,1569


剩余补采请求： 119
原数据未覆盖；刷新记录和后续日期记录分表保留，等待来源一致性与行程核验。
